In [ ]:
import functools as ft
import warnings

import jax
import jax.numpy as jnp
import jax.random as jr
import seaborn as sns
from jaxtyping import Float, Key, Scalar, ScalarLike
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
from typinox import Vmapped

import fairsim
from fairsim.model.continuous_individual import Individual, IndividualParams
from fairsim.util import KeyGen, typed

warnings.filterwarnings("ignore", "A JAX array is being set as static!")

In [ ]:
sns.set_theme(context="notebook", style="whitegrid")

In [ ]:
from fairsim.model.continuous_individual import Individual


params = IndividualParams(
    decline_credit_penalty=0.00,
    score_alpha=0.33,
    decline_latent_penalty=0.0,
    default_latent_penalty=0.33,
    repay_latent_gain=0.15,
    latent_noise_scale=0.05,
    score_noise_scale=0.05,
)

n_population = 1000
kg = KeyGen(seed=42)
score_init_noise = 0.3
probits = jr.normal(kg(), (n_population,)) * 0.8
scores = jax.scipy.stats.norm.cdf(
    probits + jr.normal(kg(), (n_population,)) * score_init_noise
)
latents = jax.scipy.stats.norm.cdf(probits)
individuals: Individual = jax.vmap(ft.partial(Individual, params=params))(latent=latents, score=scores)

In [ ]:
sns.scatterplot(x=individuals.latent, y=individuals.score)

In [ ]:
def sim_max_score_percent(individuals: Vmapped[Individual, "n_population"], threshold: Float[ScalarLike, ""], key: Key[Scalar, ""]) -> Vmapped[Individual, "n_population"]:
    key2, key3 = jr.split(key, 2)
    # threshold = jnp.quantile(individuals.score, 1 - q_top, method="nearest")
    approval = (individuals.score >= threshold).astype(float)
    prob_repay = jax.vmap(lambda ind: ind.repay_probability)(individuals)
    repayment = jr.bernoulli(key2, prob_repay).astype(float)
    noise = jr.normal(key3, individuals.latent.shape + (2,))
    @typed
    def update_individual(ind: Individual, app: Float[Scalar, ""], rep: Float[Scalar, ""], noi: Float[Scalar, "2"]):
        return ind.update(app, rep, noi)
    individuals = jax.vmap(update_individual)(individuals, approval, repayment, noise)
    return individuals

In [ ]:
n_population = 10000
kg = KeyGen(seed=42)
score_init_noise = 0.3
probits = jr.normal(kg(), (n_population,)) * 0.8
scores = jax.scipy.stats.norm.cdf(
    probits + jr.normal(kg(), (n_population,)) * score_init_noise
)
latents = jax.scipy.stats.norm.cdf(probits)
individuals: Individual = jax.vmap(ft.partial(Individual, params=params))(latent=latents, score=scores)

sim_kg = KeyGen(seed=210)
ckpt = {0: individuals}
# ckpt_indices = [0, 1, 3, 10, 30, 100, 1000, 10000, 100000]
ckpt_indices = [0, 1, 3, 10, 30, 100, 300, 1000, 3000]
last_t = 0
for t in tqdm(ckpt_indices[1:]):
    def iter(c, a):
        return sim_max_score_percent(c, threshold=0.6, key=a), None
    individuals = jax.lax.scan(iter, individuals, jax.random.split(sim_kg(), t - last_t))[0]
    individuals.latent.block_until_ready()
    last_t = t
    if t in ckpt_indices:
        ckpt[t] = individuals

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(8, 8), layout="constrained")
axes = axes.flatten()

for (k, v), ax in zip(ckpt.items(), axes):
    # sns.scatterplot(x=v.latent, y=v.score, ax=ax, alpha=0.03)
    sns.histplot(x=v.latent, ax=ax, bins=20)
    ax.set_title(f"Step {k}")

In [ ]:
sim_max_score_percent(individuals, threshold=0.6, key=kg())